In [5]:
import os
import glob
import shutil
import pandas as pd
from pathlib import Path

### Move .yml file to directories

In [ ]:
yaml_path = r"\\allen\aind\scratch\ophys\Andrew\metadata\device.yml"
basepath = r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\ASAP7\826031'
data_paths = glob.glob(os.path.join(basepath,'**','VCO1_Behavior.harp'),recursive=True)
data_paths

In [ ]:
for path in data_paths:
    if 'device.yml' in os.listdir(path):
        print(path)
        print('device metadata in path')
    else:
        shutil.copy(yaml_path,path)
        new_path = glob.glob(os.path.join(path,'**.yml'))[0]
        print(new_path)

### Move bonsai_event_log.csv to processing directories

In [ ]:
for path in data_paths:
    if 'bonsai_event_log.csv' in os.listdir(path):
        print(path)
        print('Behavior data in path')
    else:
        try:
            parent = Path(path).parents[0]
            behavior_path  = glob.glob(os.path.join(parent,'bonsai_event_log.csv'))[0]
            shutil.move(behavior_path,path)
            new_path = glob.glob(os.path.join(path,'**.csv'))[0]
            print(new_path)
        except:
            print(path)

### Move instrument.json to session_directories

In [6]:
basepath = r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
json_path = r"\\allen\aind\scratch\ophys\Andrew\metadata\instrument.json"

In [7]:
mice = [803496,804730,804733,810196,809043,809044,809047,803121,814591,814593,
        825854,825855,824806,826031,826032,826033,826034,834788,
        838410,836173,836174
]

In [30]:
from __future__ import annotations

import re
import shutil
from pathlib import Path

def find_session_root_subdir(session_dir: Path, mouse: int | str) -> Path:
    """
    Find the subdirectory inside `session_dir` whose name matches:
        nnnnnn_yyyy-mm-dd_hh-mm-ss
    where nnnnnn == mouse id (6 digits).
    Returns the matching Path, or raises ValueError with a helpful message.
    """
    mouse = str(mouse)
    pattern = re.compile(rf"^{re.escape(mouse)}_\d{{4}}-\d{{2}}-\d{{2}}_\d{{2}}-\d{{2}}-\d{{2}}$")

    matches = [p for p in session_dir.iterdir() if p.is_dir() and pattern.match(p.name)]
    if len(matches) == 0:
        raise ValueError(f"No matching destination subdir in {session_dir}")
    if len(matches) > 1:
        raise ValueError(f"Multiple matching destination subdirs in {session_dir}: {[m.name for m in matches]}")
    return matches[0]

def move_instrument_json(session_dir: Path, mouse: int | str, overwrite: bool = False) -> Path:
    """
    Move session_dir/instrument.json into the matched subdirectory.
    Returns destination path.
    """
    src = session_dir / "instrument.json"
    if not src.exists():
        raise FileNotFoundError(f"Missing {src}")

    dest_dir = find_session_root_subdir(session_dir, mouse)
    dest = dest_dir / "instrument.json"

    if dest.exists():
        if overwrite:
            dest.unlink()
        else:
            raise FileExistsError(f"Destination already has instrument.json: {dest}")

    dest_dir.mkdir(parents=True, exist_ok=True)
    shutil.move(str(src), str(dest))
    return dest

In [31]:
for mouse in mice:
    mouse = str(mouse)

    # Find the mouse directory robustly (no glob()[0] crashes)
    candidates = list(Path(basepath).rglob(mouse))
    candidates = [p for p in candidates if p.is_dir()]

    if not candidates:
        print(f"[SKIP] mouse {mouse}: no directory found under {basepath}")
        continue
    if len(candidates) > 1:
        # pick the shortest path as a heuristic; adjust if you have a better rule
        candidates = sorted(candidates, key=lambda p: len(str(p)))
        print(f"[WARN] mouse {mouse}: multiple dirs found, using {candidates[0]}")
    data_dir = candidates[0]

    session_folders = [
        p for p in data_dir.iterdir()
        if p.is_dir()
        and "processed_data" not in p.name
        and "predictive_processing_data" not in p.name
    ]

    for session_dir in session_folders:
        src = session_dir / "instrument.json"

        # your original conditional, but safer
        if not src.exists():
            continue
        if (session_dir / "behavior").exists():
            continue

        try:
            dest = move_instrument_json(session_dir, mouse, overwrite=False)
            print(f"[OK] moved {src.name} -> {dest}")
        except Exception as e:
            print(f"[ERROR] {session_dir}: {e}")

[OK] moved instrument.json -> \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-01_803496\803496_2025-07-01_14-56-10\instrument.json
[ERROR] \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-25_803496: No matching destination subdir in \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-25_803496
[ERROR] \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-28_803496: No matching destination subdir in \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-28_803496
[OK] moved instrument.json -> \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-29_803496\803496_2025-07-29_13-34-35\instrument.json
[OK] moved instrument.json -> \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-30_803496\803496_2025-07-30_10-05-23\instrument.json
[OK] moved instrument.json -> \\allen\aind\scratch\

[OK] moved instrument.json -> \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\NM\825855\2026-01-12_825855\825855_2026-01-12_11-56-04\instrument.json
[OK] moved instrument.json -> \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\NM\824806\2026-01-08_824806\824806_2026-01-08_11-12-01\instrument.json
[OK] moved instrument.json -> \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\2025-12-01_826033\826033_2025-12-01_10-44-52\instrument.json
[OK] moved instrument.json -> \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826034\2025-12-01_826034\826034_2025-12-01_11-32-40\instrument.json
[ERROR] \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\834788\834788_2026-02-13_15-22-44: No matching destination subdir in \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\834788\834788_2026-02-13_15-22-44
[ERROR] \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\8

In [ ]:
session_folders = []

for mouse in mice:
    data_dir = glob.glob(os.path.join(basepath,'**',str(mouse)))[0]
    for path in os.listdir(data_dir):
        session_folder = os.path.join(data_dir,path)
        if os.path.isdir(session_folder) and 'processed_data' not in session_folder and 'predictive_processing_data' not in session_folder:
            print(session_folder)
            session_folders.append(session_folder)

In [ ]:
for folder in session_folders:
    static_data = glob.glob(os.path.join(Path(folder).parent,'**','**VasMap**'),recursive=True)
    print(static_data)

In [ ]:
for mouse in mice:
    data_dir = glob.glob(os.path.join(basepath,'**',str(mouse)))[0]
    vas_file = glob.glob(os.path.join(data_dir,'**','**local_vasculature**'),recursive=True)
    print(vas_file)

In [ ]:
df = pd.DataFrame(columns = ['session_id',
                             'subject_id',
                             'session_#',
                             'session_date',
                             'indicator1',
                             'indicator2',
                             'dmd1_depth',
                             'dmd2_depth',
                             'paradigm',
                             'session_type',
                             'stimulus',
                             'experimentor',
                             'imaging_rig',
                             'behavior_rig',
                             'session_dir',
                             'purpose',
                             'notes'
])


In [ ]:
df['session_dir'] = session_folders

In [ ]:
df['session_id'] = [Path(path).stem for path in session_folders]

In [ ]:
import re
from datetime import datetime
from typing import Optional, Dict, Any

# Matches:
#   yyyy-mm-dd_nnnnnn
#   nnnnnn_yyyy-mm-dd
#   nnnnnn_yyyy-mm-dd_hh-mm-ss
_PAT = re.compile(
    r"""
    ^(?:
        (?P<date1>\d{4}-\d{2}-\d{2})_(?P<sid1>\d{6})               # yyyy-mm-dd_nnnnnn
        (?:_(?P<big1>Big[A-Za-z]+))?                               # optional _Big...
      |
        (?P<sid2>\d{6})_(?P<date2>\d{4}-\d{2}-\d{2})               # nnnnnn_yyyy-mm-dd ...
        (?:_(?P<time2>\d{2}-\d{2}-\d{2}))?                         # optional _hh-mm-ss
        (?:_(?P<big2>Big[A-Za-z]+))?                               # optional _Big...
    )$
    """,
    re.VERBOSE,
)

def parse_subject_and_date(s: str) -> Dict[str, Any]:
    """
    Parses:
      - yyyy-mm-dd_nnnnnn
      - yyyy-mm-dd_nnnnnn_BigVolume / _BigStacks (or any _Big<letters>)
      - nnnnnn_yyyy-mm-dd
      - nnnnnn_yyyy-mm-dd_hh-mm-ss
      - (optionally) ..._BigSomething at the end

    Returns:
      {
        'subject_id': str (6 digits),
        'date': datetime.date,
        'datetime': datetime | None,   # present if time included
      }
    """
    m = _PAT.match(s.strip())
    if not m:
        raise ValueError(f"Unrecognized format: {s!r}")

    subject_id = m.group("sid1") or m.group("sid2")
    date_str   = m.group("date1") or m.group("date2")
    time_str   = m.group("time2")  # only possible in sid2/date2 branch

    d = datetime.strptime(date_str, "%Y-%m-%d").date()

    dt: Optional[datetime] = None
    if time_str is not None:
        dt = datetime.strptime(f"{date_str}_{time_str}", "%Y-%m-%d_%H-%M-%S")

    return {"subject_id": subject_id, "date": d, "datetime": dt}

In [ ]:
for session in df['session_id'].values:
    parse = parse_subject_and_date(session)
    print(parse['date'])

In [ ]:


df['session_date'] = 
df['subject_id'] = 

In [ ]:
filen = '2026-02-16_session_df.csv'
savepath = os.path.join(basepath,filen) 
df.to_csv(savepath)

In [ ]:
Path(path).stem